# M20 · Fine-tuning / distillation

_Curriculum · Domain 4 · GenAI_

We train a small teacher classifier for ad-text intent, then train a compact student on the teacher's soft labels. The math idea is $KL(q_T \Vert p_T)$: the student learns the full teacher distribution, not just the winning class.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(20)

## Synthetic AFP-AI classification task

Think of each row as a creative or video candidate with text, visual, and account features. The label is a small taxonomy class for routing or guidance.

In [ ]:
X, y = make_classification(
    n_samples=2400,
    n_features=12,
    n_informative=7,
    n_redundant=2,
    n_classes=3,
    class_sep=2.0,
    random_state=20
)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=20,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

print(X_train.shape)
print(np.bincount(y_train))

## Teacher model

The teacher is allowed to use all 12 features. In a production system this could be a large multilingual encoder or LLM-derived classifier.

In [ ]:
teacher = LogisticRegression(
    max_iter=1000,
    C=3.0
)

teacher.fit(X_train, y_train)

teacher_val = teacher.predict_proba(X_val)
teacher_pred = teacher_val.argmax(axis=1)
teacher_acc = accuracy_score(y_val, teacher_pred)

print(round(teacher_acc, 3))
assert teacher_acc > 0.75

## Student from hard labels

The student sees only the first four features, mimicking a cheaper serving model. First we train it in the ordinary supervised way.

In [ ]:
student_hard = LogisticRegression(
    max_iter=1000,
    C=1.0
)

student_hard.fit(X_train[:, :4], y_train)

hard_val = student_hard.predict_proba(X_val[:, :4])
hard_pred = hard_val.argmax(axis=1)
hard_acc = accuracy_score(y_val, hard_pred)

print(round(hard_acc, 3))

## Soft labels for distillation

Temperature makes the teacher less certain, so near-miss classes still teach the student. We use $q_T(c)=\operatorname{softmax}(z_c/T)$ conceptually; here probabilities are softened by raising them to $1/T$ and renormalizing.

In [ ]:
def soften(probabilities, temperature):
    adjusted = probabilities ** (1.0 / temperature)
    adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)
    return adjusted

T = 2.0
teacher_train = teacher.predict_proba(X_train)
soft_targets = soften(teacher_train, T)

row_sum = soft_targets[0].sum()
print(np.round(soft_targets[0], 3))
assert np.isclose(row_sum, 1.0)

## Distilled student via sample weights

Scikit-learn does not train directly on soft multiclass labels, so we expand each row once per class with a sample weight equal to the teacher probability. This optimizes the same cross-entropy target.

In [ ]:
classes = np.arange(3)
X_small = X_train[:, :4]
X_distill = np.repeat(X_small, repeats=3, axis=0)
y_distill = np.tile(classes, X_small.shape[0])
w_distill = soft_targets.reshape(-1)

student_soft = LogisticRegression(
    max_iter=1000,
    C=1.0
)

student_soft.fit(X_distill, y_distill, sample_weight=w_distill)

soft_val = student_soft.predict_proba(X_val[:, :4])
soft_pred = soft_val.argmax(axis=1)
soft_acc = accuracy_score(y_val, soft_pred)

print(round(soft_acc, 3))

## Compare teacher, hard student, and distilled student

The distilled student may or may not beat hard-label training on this small synthetic task, but it should learn a valid probability distribution and often improves calibration-like behavior.

In [ ]:
rows = [
    {"model": "teacher", "accuracy": teacher_acc, "log_loss": log_loss(y_val, teacher_val)},
    {"model": "hard_student", "accuracy": hard_acc, "log_loss": log_loss(y_val, hard_val)},
    {"model": "distilled_student", "accuracy": soft_acc, "log_loss": log_loss(y_val, soft_val)}
]

results = pd.DataFrame(rows)
print(results.round(3))
assert np.allclose(soft_val.sum(axis=1), 1.0)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(results["model"], results["accuracy"], color=["#4c78a8", "#f58518", "#54a24b"])
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("validation accuracy")
ax.set_title("teacher and students")
plt.xticks(rotation=20)
plt.show()

## Takeaway

Fine-tuning adapts behavior. Distillation transfers that behavior into a smaller model. LoRA sits between them by learning only a low-rank update $\Delta W=BA$ instead of every weight.